# 00_환경설정

In [1]:
import os
import sys

import glob
import re

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

print(f"sys.platform: {sys.platform}")
print(f"sys.executable: {sys.executable}")
print(f"os.getcwd(): {os.getcwd()}")
print(f"TensorFlow version: {tf.__version__}")

# Pandas 출력 옵션
pd.set_option('display.max_rows', None)


I0000 00:00:1773645201.091060  176758 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


sys.platform: linux
sys.executable: /usr/bin/python3
os.getcwd(): /home/jovyan/d/git/01_mdls_ds8_open/08_dl/260315_dl_lyricist
TensorFlow version: 2.21.0


In [ ]:
# GPU 확인 및 기본 최적화
# TensorFlow가 현재 머신에서 인식하는 물리 장치 목록 조회
# 반환값은 리스트이며, 장치가 없으면 빈 리스트 반환 
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')
print("CPUs:", cpus)
print("GPUs:", gpus)


# GPU가 1개 이상 인식된 경우에만 GPU 관련 최적화 설정 수행
USE_MIXED_PRECISION = False # True로 바꾸면 mixed precision 사용
if gpus:
    try:
        for gpu in gpus:     # GPU 메모리 점유를 필요 시점으로 미룸
            tf.config.experimental.set_memory_growth(gpu, True)
            # GPU 메모리 증가 방식(memory growth) 활성화
            # 이 옵션을 켜면 필요한 만큼만 점진적으로 메모리를 사용
        print("GPU memory growth enabled")
    except Exception as e:
        print("GPU memory growth setting failed:", e)


    try:
        from tensorflow.keras import mixed_precision
        # Mixed Precision(혼합 정밀도) 설정
        # 현재는 NaN 방지를 위해 float32 우선
        policy_name = 'mixed_float16' if USE_MIXED_PRECISION else 'float32'
        mixed_precision.set_global_policy(policy_name)
        print(f"Precision policy: {policy_name}")
    except Exception as e:
        print("Precision policy setting failed:", e)
else:
    print("GPU not detected. Running on CPU.")

AUTOTUNE = tf.data.AUTOTUNE
# tf.data 입력 파이프라인 최적화용 상수
# AUTOTUNE을 사용하면 TensorFlow가 prefetch / map / interleave 등의 병렬 처리 수준을 자동으로 튜닝함
# 직접 숫자를 지정하는 대신, 런타임이 적절한 값을 잡도록 맡기는 옵션


CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled
Precision policy: float32


W0000 00:00:1773645204.248689  176758 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


# 01_데이터 읽어오기

- glob 모듈을 사용하면 파일을 읽어오는 작업을 하기가 아주 용이
- glob 를 활용하여 모든 txt 파일을 읽어온 후, raw_corpus 리스트에 문장 단위로 저장

In [3]:
import glob #glob 모듈의 glob 함수는 사용자가 제시한 조건에 맞는 파일명을 리스트 형식으로 반환한다
import os

txt_file_path = '/home/jovyan/d/git/01_mdls_ds8_open/08_dl/260315_dl_lyricist/00_data/*'
txt_list = glob.glob(txt_file_path)

raw_corpus = []

for txt_file in txt_list:
    with open(txt_file, "r", encoding="utf-8") as f:
        raw = f.read().splitlines()
        raw_corpus.extend(raw)

print("데이터 크기:", len(raw_corpus))
print("Examples:", raw_corpus[:3])

데이터 크기: 187088
Examples: ['Looking for some education', 'Made my way into the night', 'All that bullshit conversation']


In [4]:
raw_corpus

['Looking for some education',
 'Made my way into the night',
 'All that bullshit conversation',
 "Baby, can't you read the signs? I won't bore you with the details, baby",
 "I don't even wanna waste your time",
 "Let's just say that maybe",
 'You could help me ease my mind',
 "I ain't Mr. Right But if you're looking for fast love",
 "If that's love in your eyes",
 "It's more than enough",
 'Had some bad love',
 "So fast love is all that I've got on my mind Ooh, ooh",
 'Ooh, ooh Looking for some affirmation',
 'Made my way into the sun',
 'My friends got their ladies',
 "And they're all having babies",
 "I just wanna have some fun I won't bore you with the details, baby",
 "I don't even wanna waste your time",
 "Let's just say that maybe",
 'You could help me ease my mind',
 "I ain't Mr. Right But if you're looking for fast love",
 "If that's love in your eyes",
 "It's more than enough",
 "I've had some bad love",
 "So fast love is all that I've got on my mind Ooh, ooh",
 'Baby, baby',

# 02_데이터 정제

- 앞서 배운 테크닉들을 활용해 문장 생성에 적합한 모양새로 데이터를 정제
- preprocess_sentence() 함수를 만든 것을 통해 데이터를 정제
- 추가로 지나치게 긴 문장은 다른 데이터들이 과도한 Padding을 갖게 하므로 제거
    - 문장을 토큰화 했을 때 토큰의 개수가 15개를 넘어가는 문장을 학습 데이터에서 제외  

In [5]:
import re

# raw_corpus -> 점검용 으로 저장
s = pd.Series(raw_corpus, name="text")

In [6]:

print("\n","기본 정보")
print("총 줄 수:", len(s))
print("dtype:", s.dtype)

print("\n 빈 줄 / 공백 줄")
empty_mask = s.isna() | (s.str.strip() == "") 
print("빈 줄 수:", int(empty_mask.sum()))
print("빈 줄 비율:", round(empty_mask.mean(), 4))
# empty_mask = "이 줄이 비어 있는가?"를 True/False로 저장한 불리언 마스크
# | 는 OR(또는) 연산
# - 둘 중 하나라도 True면 True
# - 즉,
#   1) 결측값이거나
#   2) 공백만 있는 문자열이면
#   "빈 줄"로 판단

print("\n길이 분포")
lengths = s.fillna("").str.len() # 결측값(NaN, None 등)을 빈 문자열 ""로 바꾸고 계산
print(lengths.describe())

print("\n너무 긴 줄(상위 10개):")
print(s.iloc[lengths.sort_values(ascending=False).head(10).index].to_list())


 기본 정보
총 줄 수: 187088
dtype: object

 빈 줄 / 공백 줄
빈 줄 수: 11128
빈 줄 비율: 0.0595

길이 분포
count    187088.000000
mean         34.977070
std          21.546886
min           0.000000
25%          22.000000
50%          33.000000
75%          46.000000
max        1465.000000
Name: text, dtype: float64

너무 긴 줄(상위 10개):
["WRITERS RUSSELL BROWN, IRWIN LEVINE I'm comin' home, I've done my time Now I've got to know what is and isn't mine If you received my letter telling you I'd soon be free Then you'll know just what to do if you still want me If you still want me Just tie a yellow ribbon 'round the old oak tree It's been way too long, do you still want me? If I don't see a ribbon 'round the old oak tree I'll just stay on the bus, forget about us, put the blame on me If I don't see a yellow ribbon 'round the old oak tree Bus driver, please look for me 'Cause I couldn't bear to see what I might see I'm really still in prison and my love, he holds the key A simple yellow ribbon's all I need to set m

In [7]:
def preprocess_sentence(sentence):
    """
    하나의 원문 문장을 받아서
    모델 학습에 넣기 좋은 형태로 정제한 뒤 반환
    """
    sentence = str(sentence).lower().strip()
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = re.sub(r"[^a-zA-Z'?.!,¿]+", " ", sentence)
    sentence = sentence.strip()
    return sentence


corpus_rev = []
remove_duplicates = False   # 필요하면 True로 가능


for sentence in s:
    if pd.isna(sentence):
        continue
    sentence = str(sentence).strip()
    if len(sentence) == 0:
        continue
    if sentence.endswith(":"):
        continue

    cleaned = preprocess_sentence(sentence)
    raw_tok_len = len(cleaned.split())   # <start>, <end> 제외
    if 2 <= raw_tok_len <= 15:
        corpus_rev.append(f"<start> {cleaned} <end>")

# 완전 동일 문장 중복 제거
if remove_duplicates:
    corpus_rev = list(dict.fromkeys(corpus_rev))

print("정제 후 총 줄 수:", len(corpus_rev))
print("정제 샘플:", corpus_rev[:5])

정제 후 총 줄 수: 162919
정제 샘플: ['<start> looking for some education <end>', '<start> made my way into the night <end>', '<start> all that bullshit conversation <end>', "<start> i don't even wanna waste your time <end>", "<start> let's just say that maybe <end>"]


In [8]:
corpus_rev

['<start> looking for some education <end>',
 '<start> made my way into the night <end>',
 '<start> all that bullshit conversation <end>',
 "<start> i don't even wanna waste your time <end>",
 "<start> let's just say that maybe <end>",
 '<start> you could help me ease my mind <end>',
 "<start> i ain't mr . right but if you're looking for fast love <end>",
 "<start> if that's love in your eyes <end>",
 "<start> it's more than enough <end>",
 '<start> had some bad love <end>',
 "<start> so fast love is all that i've got on my mind ooh , ooh <end>",
 '<start> ooh , ooh looking for some affirmation <end>',
 '<start> made my way into the sun <end>',
 '<start> my friends got their ladies <end>',
 "<start> and they're all having babies <end>",
 "<start> i just wanna have some fun i won't bore you with the details , baby <end>",
 "<start> i don't even wanna waste your time <end>",
 "<start> let's just say that maybe <end>",
 '<start> you could help me ease my mind <end>',
 "<start> i ain't mr 

# 03_토큰화 + 평가 데이터셋 분리

- tokenize() 함수로 데이터를 Tensor로 변환
- sklearn 모듈의 train_test_split() 함수를 사용해 훈련 데이터와 평가 데이터를 분리
- 단어장의 크기는 12,000 이상 으로 설정, 총 데이터의 20% 를 평가 데이터셋으로 사용

In [9]:
from sklearn.model_selection import train_test_split

# 먼저 텍스트 기준으로 train/val 분리
train_texts, val_texts = train_test_split(
    corpus_rev,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# tokenizer는 train_texts로만 학습
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=12000,
    filters=' ',
    oov_token="<unk>"
)
tokenizer.fit_on_texts(train_texts)

# train/val 각각 시퀀스로 변환
train_tensor = tokenizer.texts_to_sequences(train_texts)
val_tensor = tokenizer.texts_to_sequences(val_texts)

# 패딩 / 자르기
MAX_LEN = 17  # 원문 15 + <start> + <end>
train_tensor = tf.keras.preprocessing.sequence.pad_sequences(
    train_tensor,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

val_tensor = tf.keras.preprocessing.sequence.pad_sequences(
    val_tensor,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)
# 입력(X), 타깃(y)
enc_train = train_tensor[:, :-1]
dec_train = train_tensor[:, 1:]

enc_val = val_tensor[:, :-1]
dec_val = val_tensor[:, 1:]

# 패딩(0) 위치는 loss 계산에서 제외하기 위한 가중치
train_sw = (dec_train != 0).astype("float32")
val_sw = (dec_val != 0).astype("float32")

print("len(train_texts):", len(train_texts))
print("len(val_texts):", len(val_texts))
print("enc_train shape:", enc_train.shape)
print("dec_train shape:", dec_train.shape)
print("enc_val shape:", enc_val.shape)
print("dec_val shape:", dec_val.shape)

len(train_texts): 130335
len(val_texts): 32584
enc_train shape: (130335, 16)
dec_train shape: (130335, 16)
enc_val shape: (32584, 16)
dec_val shape: (32584, 16)


# 05_모델 설계 및 학습

## 05-1_데이터셋 최적화

In [10]:
# 단어장 크기 계산
if tokenizer.num_words is not None:
    vocab_size = tokenizer.num_words
else:
    vocab_size = len(tokenizer.word_index) + 1

vocab_size = vocab_size + 1
print("vocab_size:", vocab_size)


vocab_size: 12001


In [ ]:

# 배치 크기
BATCH_SIZE = 32

train_ds = tf.data.Dataset.from_tensor_slices((enc_train, dec_train, train_sw))
train_ds = (
    train_ds
    .shuffle(buffer_size=min(len(enc_train), 50000), reshuffle_each_iteration=True)
    .batch(BATCH_SIZE, drop_remainder=False)
    .cache()
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((enc_val, dec_val, val_sw))
val_ds = (
    val_ds
    .batch(BATCH_SIZE, drop_remainder=False)
    .cache()
    .prefetch(AUTOTUNE)
)

# prefetch(AUTOTUNE)
# 모델이 현재 배치를 학습하는 동안 다음 배치를 백그라운드에서 미리 준비
# 입력 파이프라인 병목을 줄여 GPU/CPU 유휴 시간을 감소시킴
# AUTOTUNE: 몇 개 배치를 미리 가져올지 TensorFlow가 자동 판단

W0000 00:00:1773645207.344403  176758 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1773645207.474625  176758 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13358 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a


## 05-3_모델생성

- 모델의 Embedding Size와 Hidden Size를 조절, 10 Epoch 안에 val_loss 값을 2.2 수준으로 줄일 수 있는 모델 설계

In [12]:
# 실험 설정값 정리
# 모델 생성부와 로그 기록부가 어긋나지 않도록 실제 사용한 값을 명시적으로 정리
embedding_size = 256
hidden_size = 1024

dropout_rate = 0.2
recurrent_dropout_rate = 0.0
learning_rate_start = 5e-4
clipnorm_value = 1.0

# 모델 설계
lyricist = tf.keras.Sequential([
    tf.keras.Input(shape=(enc_train.shape[1],)),
    tf.keras.layers.Embedding(vocab_size, embedding_size, mask_zero=True),
    tf.keras.layers.SpatialDropout1D(dropout_rate),

    tf.keras.layers.LSTM(
        hidden_size,
        return_sequences=True,
        dropout=dropout_rate,
        recurrent_dropout=recurrent_dropout_rate ),
    tf.keras.layers.LSTM(
        hidden_size,
        return_sequences=True,
        dropout=dropout_rate,
        recurrent_dropout=recurrent_dropout_rate),
    tf.keras.layers.Dense(vocab_size, dtype='float32')])

loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction='none')

optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate_start,
    clipnorm=clipnorm_value)


# 모델 컴파일 / 개요로 구조 확인
lyricist.compile(optimizer=optimizer,
                 loss=loss)
lyricist.summary()

# 콜백 설정 
callbacks = [
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        start_from_epoch=4)
        ]

# 학습 수행
history = lyricist.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1)

print("최저 val_loss:", float(np.min(history.history["val_loss"])))
print("마지막 val_loss:", float(history.history["val_loss"][-1]))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 16, 256)        │     3,072,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 16, 256)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16, 1024)       │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16, 1024)       │     8,392,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16, 12001)      │    12,301,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,012,961 (110.68 MB)

 Trainable params: 29,012,961 (110.68 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


I0000 00:00:1773645211.964204  176924 cuda_dnn.cc:461] Loaded cuDNN version 91002


4073/4073 ━━━━━━━━━━━━━━━━━━━━ 247s 60ms/step - loss: 2.9291 - val_loss: 2.7235 - learning_rate: 5.0000e-04
Epoch 2/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 246s 60ms/step - loss: 2.6393 - val_loss: 2.5859 - learning_rate: 5.0000e-04
Epoch 3/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 281s 69ms/step - loss: 2.4700 - val_loss: 2.5077 - learning_rate: 5.0000e-04
Epoch 4/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 231s 57ms/step - loss: 2.3259 - val_loss: 2.4647 - learning_rate: 5.0000e-04
Epoch 5/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 223s 55ms/step - loss: 2.2023 - val_loss: 2.4378 - learning_rate: 5.0000e-04
Epoch 6/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 225s 55ms/step - loss: 2.0951 - val_loss: 2.4178 - learning_rate: 5.0000e-04
Epoch 7/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 240s 59ms/step - loss: 1.9994 - val_loss: 2.4024 - learning_rate: 5.0000e-04
Epoch 8/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 231s 57ms/step - loss: 1.9115 - val_loss: 2.3920 - learning_rate: 5.0000e-04
Epoch 9/10
4073/4073 ━━━━━━━━━━━━━━━━━━━━ 226s 56ms/step - 

# 06_실험 결과 기록

In [13]:
from pathlib import Path
from datetime import datetime

# 저장 폴더
log_dir = Path("./experiment_logs")
log_dir.mkdir(parents=True, exist_ok=True)

# 현재 시각 / 실행 ID
run_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# 이번 실험 설정값 요약
run_config = {
    "run_id": run_id,
    "run_time": run_time,
    "remove_duplicates": remove_duplicates,
    "max_len": MAX_LEN,
    "vocab_size": vocab_size,
    "tokenizer_num_words": tokenizer.num_words,
    "batch_size": BATCH_SIZE,
    "embedding_size": embedding_size,
    "hidden_size": hidden_size,
    "dropout": dropout_rate,
    "recurrent_dropout": recurrent_dropout_rate,
    "learning_rate_start": learning_rate_start,
    "clipnorm": clipnorm_value,
    "train_size": len(train_texts),
    "val_size": len(val_texts),
    "train_tensor_shape": str(enc_train.shape),
    "val_tensor_shape": str(enc_val.shape),
}

# history에서 learning_rate 컬럼 이름 찾기
lr_key = None
for k in history.history.keys():
    if k in ["learning_rate", "lr"]:
        lr_key = k
        break

# epoch별 기록 테이블 생성
epoch_rows = []
num_epochs = len(history.history["loss"])

for i in range(num_epochs):
    row = run_config.copy()
    row["epoch"] = i + 1
    row["loss"] = float(history.history["loss"][i])
    row["val_loss"] = float(history.history["val_loss"][i])

    # learning rate 기록
    if lr_key is not None:
        row["learning_rate"] = float(history.history[lr_key][i])
    else:
        row["learning_rate"] = np.nan

    epoch_rows.append(row)
epoch_df = pd.DataFrame(epoch_rows)


# 요약 기록 생성
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

summary_row = run_config.copy()
summary_row["epochs_ran"] = num_epochs
summary_row["best_epoch"] = best_epoch
summary_row["best_loss"] = float(np.nanmin(history.history["loss"]))
summary_row["best_val_loss"] = float(np.nanmin(history.history["val_loss"]))
summary_row["last_loss"] = float(history.history["loss"][-1])
summary_row["last_val_loss"] = float(history.history["val_loss"][-1])

if lr_key is not None:
    summary_row["last_learning_rate"] = float(history.history[lr_key][-1])
else:
    summary_row["last_learning_rate"] = np.nan

summary_df = pd.DataFrame([summary_row])


# 저장 파일 경로
epoch_log_path = log_dir / "training_history_log.csv"
summary_log_path = log_dir / "training_summary_log.csv"


# 기존 파일 있으면 append
if epoch_log_path.exists():
    old_epoch_df = pd.read_csv(epoch_log_path)
    epoch_df = pd.concat([old_epoch_df, epoch_df], ignore_index=True)

if summary_log_path.exists():
    old_summary_df = pd.read_csv(summary_log_path)
    summary_df = pd.concat([old_summary_df, summary_df], ignore_index=True)


# 저장
epoch_df.to_csv(epoch_log_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_log_path, index=False, encoding="utf-8-sig")

print(f"[저장 완료] epoch 로그: {epoch_log_path}")
print(f"[저장 완료] summary 로그: {summary_log_path}")

print("\n[이번 실험 요약]")
summary_df.tail(1)

[저장 완료] epoch 로그: experiment_logs/training_history_log.csv
[저장 완료] summary 로그: experiment_logs/training_summary_log.csv

[이번 실험 요약]


,run_id,run_time,remove_duplicates,vocab_size,tokenizer_num_words,batch_size,embedding_size,hidden_size,dropout,recurrent_dropout,...,val_tensor_shape,epochs_ran,best_epoch,best_val_loss,last_val_loss,last_loss,max_len,clipnorm,best_loss,last_learning_rate
3,20260316_075308,2026-03-16 07:53:08,False,12001,12000,32,256,1024,0.2,0.0,...,"(32584, 16)",10,10,2.383806,2.383806,1.757603,17.0,1.0,1.757603,0.0005


# 06-1_최근 실험 비교 보기

In [14]:
summary_log_path = Path("./experiment_logs/training_summary_log.csv")

if summary_log_path.exists():
    exp_summary = pd.read_csv(summary_log_path)
    display(
        exp_summary.sort_values(
            by=["best_val_loss", "run_time"],
            ascending=[True, False]
        ).reset_index(drop=True)
    )
else:
    print("파일 없음")

,run_id,run_time,remove_duplicates,vocab_size,tokenizer_num_words,batch_size,embedding_size,hidden_size,dropout,recurrent_dropout,...,val_tensor_shape,epochs_ran,best_epoch,best_val_loss,last_val_loss,last_loss,max_len,clipnorm,best_loss,last_learning_rate
0,20260316_075308,2026-03-16 07:53:08,False,12001,12000,32,256,1024,0.2,0.0,...,"(32584, 16)",10,10,2.383806,2.383806,1.757603,17.0,1.0,1.757603,0.0005
1,20260316_062551,2026-03-16 06:25:51,False,12001,12000,32,256,768,0.1,0.0,...,"(31250, 14)",10,10,2.706267,2.706267,1.911067,NaN,NaN,NaN,NaN
2,20260316_062428,2026-03-16 06:24:28,False,12001,12000,32,256,768,0.1,0.0,...,"(31250, 14)",10,10,2.706267,2.706267,1.911067,NaN,NaN,NaN,NaN
3,20260316_055528,2026-03-16 05:55:28,True,12001,12000,16,256,768,0.1,0.0,...,"(20453, 14)",5,2,2.994598,3.099864,2.531337,NaN,NaN,NaN,NaN
